# SAAF palm counting | Al Ain experimental benchmark

Use a Google Colab GPU runtime. The public [Al Ain Municipality dataset](https://github.com/NourO93/Palm-Tree-Dataset) is for research and education only and prohibits commercial use. Confirm that your intended hackathon use complies before downloading or training. If terms or follow-on use are unclear, obtain permission from the dataset owner. Cite Al Saad et al., *Autonomous Palm Tree Detection from Remote Sensing Images – UAE Dataset*, IGARSS 2022.

This is an **experimental image-level split with unverified spatial independence**. The inspected export had 1,078 TIFFs, 922 matching XML annotations, 156 unlabeled images, and 9,874 palm boxes. The original test folder contained no usable images. The notebook derives a new split from the 922 paired images. Exact and perceptual checks catch repeated imagery, but cannot establish independence across farms, flights, or overlapping areas. Report results only for this fixed Al Ain split. They do not measure geographic generalization.


In [ ]:
# Change these only after reviewing the source terms and evaluation scope.
HACKATHON_USE_COMPLIES = False
RUN_GPU_TRAINING = False
if HACKATHON_USE_COMPLIES is not True:
    raise RuntimeError("Confirm that the intended hackathon use complies with the research/education-only terms.")

%pip install -q "ultralytics==8.3.230"

import hashlib
import json
import random
import re
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter
from io import BytesIO
from itertools import combinations
from pathlib import Path, PurePosixPath
from shutil import copyfile
from zipfile import ZipFile

import numpy as np
import torch
import ultralytics
import yaml
from IPython.display import display
from PIL import Image
from scipy.fftpack import dct
from ultralytics import YOLO

SOURCE = "https://github.com/NourO93/Palm-Tree-Dataset"
REVISION = "593352ae5f0e8fa460ad30afd13cc95c01bcc056"
WORK = Path("/content/saaf_al_ain")
WORK.mkdir(parents=True, exist_ok=True)
SEED = 2026
SPLIT_SIZES = {"train": 737, "val": 92, "test": 93}
CONFIDENCE = 0.25  # Set using validation only if tuning
IMGSZ = 640
MAX_DET = 3000


In [ ]:
def download_source():
    if HACKATHON_USE_COMPLIES is not True:
        raise RuntimeError("Confirm the intended hackathon use first.")
    commit = REVISION
    if not re.fullmatch(r"[a-f0-9]{40}", commit):
        raise ValueError("Set REVISION to the audited GitHub commit SHA.")
    url = f"https://codeload.github.com/NourO93/Palm-Tree-Dataset/zip/{commit}"
    archive = WORK / f"source-{commit}.zip"
    if not archive.exists():
        temporary = archive.with_suffix(".download")
        urllib.request.urlretrieve(url, temporary)
        temporary.replace(archive)
    return archive, commit, url

archive_path, commit, archive_url = download_source()
print(f"Al Ain input archive: {archive_path.name}")


In [ ]:
def index_members(archive, suffixes, split):
    found = {}
    for name in archive.namelist():
        path = PurePosixPath(name)
        if len(path.parts) < 3 or path.parts[1].lower() != split or path.suffix.lower() not in suffixes:
            continue
        key = path.stem.casefold()
        if key in found:
            raise ValueError(f"Ambiguous {split} filename stem: {key}")
        found[key] = name
    return found


def read_boxes(xml_bytes, image_key):
    root = ET.fromstring(xml_bytes)
    declared_size = (int(root.findtext("size/width")), int(root.findtext("size/height")))
    if declared_size != (256, 256):
        raise ValueError(f"Unexpected XML dimensions for {image_key}: {declared_size}")
    boxes = []
    for obj in root.findall("object"):
        if obj.findtext("name", "").strip() != "palmtree":
            raise ValueError(f"Unexpected object class in {image_key}")
        xmin, ymin, xmax, ymax = (
            int(obj.findtext(f"bndbox/{key}")) for key in ("xmin", "ymin", "xmax", "ymax")
        )
        if not (1 <= xmin <= xmax <= 256 and 1 <= ymin <= ymax <= 256):
            raise ValueError(f"Invalid Pascal VOC box in {image_key}")
        # Pascal VOC uses one-based inclusive pixel bounds. Convert to continuous image edges.
        boxes.append((xmin - 1, ymin - 1, xmax, ymax))
    return boxes


def phash(image):
    gray = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS))
    low_frequencies = dct(dct(gray, axis=0), axis=1)[:8, :8]
    bits = low_frequencies > np.median(low_frequencies)
    return int.from_bytes(np.packbits(bits).tobytes(), "big")


def audit_dataset(archive):
    images = index_members(archive, {".tif", ".tiff"}, "train")
    annotations = index_members(archive, {".xml"}, "train")
    original_test = index_members(archive, {".tif", ".tiff", ".jpg", ".jpeg", ".png"}, "test")
    paired = sorted(images.keys() & annotations.keys())
    if (len(images), len(annotations), len(paired), len(original_test)) != (1078, 922, 922, 0):
        raise ValueError("Archive file counts differ from the inspected Al Ain release.")
    exact_hashes, perceptual, labels = Counter(), {}, {}
    for key in paired:
        with Image.open(BytesIO(archive.read(images[key]))) as source:
            if source.format != "TIFF" or source.size != (256, 256) or source.n_frames != 1:
                raise ValueError(f"Invalid TIFF format or dimensions: {images[key]}")
            rgb = source.convert("RGB")
            exact_hashes[hashlib.sha256(rgb.tobytes()).hexdigest()] += 1
            perceptual[key] = phash(rgb)
        labels[key] = read_boxes(archive.read(annotations[key]), key)

    exact_groups = sum(count > 1 for count in exact_hashes.values())
    near_pairs = sum((left ^ right).bit_count() <= 5 for left, right in combinations(perceptual.values(), 2))
    boxes = sum(map(len, labels.values()))
    audit = {
        "tiff_images": len(images), "xml_annotations": len(annotations),
        "paired_images": len(paired), "excluded_unlabeled": len(images.keys() - annotations.keys()),
        "original_test_images": len(original_test), "boxes": boxes,
        "exact_duplicate_groups": exact_groups, "phash_pairs_at_distance_le_5": near_pairs,
        "image_size_px": [256, 256], "original_class": "palmtree",
        "spatial_independence": "unverified",
    }
    if (boxes, exact_groups, near_pairs) != (9874, 0, 0):
        raise ValueError(f"Labels or duplicate checks differ from the inspected release: {audit}")
    return images, labels, audit


def prepare_dataset(archive, images, labels, output):
    keys = sorted(labels)
    random.Random(SEED).shuffle(keys)
    offset = 0
    assignments = {"seed": SEED, "splits": {}}
    for split, count in SPLIT_SIZES.items():
        selection = keys[offset:offset + count]
        offset += count
        assignments["splits"][split] = [images[key] for key in selection]
        image_dir = output / "images" / split
        label_dir = output / "labels" / split
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        expected_images = {f"{key}.png" for key in selection}
        expected_labels = {f"{key}.txt" for key in selection}
        if ({p.name for p in image_dir.iterdir()} - expected_images
                or {p.name for p in label_dir.iterdir()} - expected_labels):
            raise ValueError("Output split contains unexpected files. Select a fresh output directory.")
        for key in selection:
            with Image.open(BytesIO(archive.read(images[key]))) as source:
                source.convert("RGB").save(image_dir / f"{key}.png", format="PNG")
            rows = []
            for x1, y1, x2, y2 in labels[key]:
                xywh = ((x1 + x2) / 512, (y1 + y2) / 512, (x2 - x1) / 256, (y2 - y1) / 256)
                rows.append("0 " + " ".join(f"{value:.9f}" for value in xywh))
            (label_dir / f"{key}.txt").write_text("\n".join(rows) + ("\n" if rows else ""))
    if offset != len(keys):
        raise ValueError("Experimental split does not consume every labeled image.")
    config = output / "data.yaml"
    # The class name is recognized by the local Monitoring Agent.
    config.write_text(yaml.safe_dump({
        "path": str(output.resolve()), "train": "images/train", "val": "images/val",
        "test": "images/test", "names": {0: "palm_tree"},
    }, sort_keys=False), encoding="utf-8")
    return config, assignments


data_root = WORK / f"yolo-{commit}-seed-{SEED}"
with ZipFile(archive_path) as archive:
    image_members, labels, audit = audit_dataset(archive)
    config, assignments = prepare_dataset(archive, image_members, labels, data_root)
audit.update(source=SOURCE, archive=archive_url, commit=commit, scope="experimental Al Ain image split")
assignments.update(commit=commit)
audit_path = data_root / "dataset_audit.json"
split_path = data_root / "split_manifest.json"
audit_path.write_text(json.dumps(audit, indent=2))
split_path.write_text(json.dumps(assignments, indent=2))
print(json.dumps(audit, indent=2))
print("Split sizes:", {split: len(paths) for split, paths in assignments["splits"].items()})


## Training and held-out evaluation

Use validation data for model selection and any threshold tuning. Freeze settings before running the held-out test. Ultralytics detection precision and recall use its metric operating point. The per-image counting MAE below uses the configured inference confidence and detection cap. It represents visible crowns in individual tiles. Evaluate local CPU performance separately if deploying the checkpoint for the permitted purpose.


In [ ]:
from google.colab import files

if HACKATHON_USE_COMPLIES is not True or RUN_GPU_TRAINING is not True or not torch.cuda.is_available():
    raise RuntimeError("Training requires confirmed permitted use, RUN_GPU_TRAINING, and a Colab GPU.")
if not 0 < CONFIDENCE <= 1:
    raise ValueError("Set a valid deployment confidence.")
model = YOLO("yolo11n.pt")
model.train(data=str(config), device=0, imgsz=IMGSZ, epochs=50, batch=8, patience=10,
            seed=SEED, workers=2, project=str(WORK / "runs"), name="al-ain-yolo11n")
best = Path(model.trainer.best)
if not best.is_file():
    raise FileNotFoundError(best)


In [ ]:
detector = YOLO(str(best))
score = detector.val(data=str(config), split="test", device=0, imgsz=IMGSZ, max_det=MAX_DET)
test_images = sorted((data_root / "images/test").glob("*.png"))
if len(test_images) != SPLIT_SIZES["test"]:
    raise ValueError("Unexpected held-out image count.")
errors, capped = [], 0
preview = []
for path in test_images:
    result, = detector.predict(str(path), device=0, imgsz=IMGSZ, conf=CONFIDENCE,
                               classes=[0], agnostic_nms=True, max_det=MAX_DET, verbose=False)
    target = len(labels[path.stem])
    predicted = len(result.boxes)
    errors.append(abs(predicted - target))
    capped += int(predicted >= MAX_DET)
    if len(preview) < 6:
        preview.append((path.name, target, predicted, Image.fromarray(result.plot()[:, :, ::-1])))

report = {
    "dataset_commit": commit, "scope": "experimental Al Ain image-level test",
    "spatial_independence": "unverified", "claims_geographic_generalization": False,
    "precision": float(score.box.mp), "recall": float(score.box.mr),
    "mAP50": float(score.box.map50), "mAP50-95": float(score.box.map),
    "count_mae_per_image": sum(errors) / len(errors), "test_images": len(errors),
    "count_capped_images": capped, "confidence": CONFIDENCE, "imgsz": IMGSZ,
    "max_detections": MAX_DET, "seed": SEED, "ultralytics": ultralytics.__version__,
    "torch": str(torch.__version__), "weights_sha256": hashlib.sha256(best.read_bytes()).hexdigest(),
}
print(json.dumps(report, indent=2))
for filename, target, predicted, annotated in preview:
    print(f"{filename} | labeled: {target} | predicted: {predicted}")
    display(annotated)


In [ ]:
export_dir = WORK / "export"
export_dir.mkdir(exist_ok=True)
weights_file = export_dir / "best.pt"
report_file = export_dir / "evaluation_report.json"
copyfile(best, weights_file)
report_file.write_text(json.dumps(report, indent=2))
copyfile(audit_path, export_dir / "dataset_audit.json")
copyfile(split_path, export_dir / "split_manifest.json")
files.download(str(weights_file))
files.download(str(report_file))
print("Audit and split manifest:", export_dir)


Keep the original archive and derived files separate. Keep downloads out of Git. For local CPU inference within permitted use, choose a trusted checkpoint, set `SAAF_PALM_WEIGHTS` to its path, and match the confidence, inference size, and detection cap to the evaluation report.